# nb03_main_analysis
**Chủ:** C · tiêu chí 5 (họ chính: Mô tả / OLAP)

**Câu hỏi:** những danh mục và đặc điểm nào (giá, dung lượng, lượt cài, độ mới của bản cập nhật) đi cùng tỉ lệ app điểm thấp cao nhất?

**Cấp phân tích: APP** (một dòng = một app sau khử trùng, `dw.fact_app`). Mọi số đều kèm *n*. Dùng "đi cùng", không dùng "gây ra".

**Giới hạn (nêu trước phân tích):** chỉ app có điểm và `store_reviews >= MIN_STORE_REVIEWS`; app không có điểm bị loại khỏi phân tích điểm; dữ liệu là snapshot đến 2018-08, không có chiều thời gian theo phiên bản; không suy ra nguyên nhân.
Điểm thấp = `Rating <= LOW_RATING_THRESHOLD`. Các band (giá, dung lượng, lượt cài, độ cũ) lấy từ các dimension của B, nên khớp các view `mart.*` dùng cho dashboard.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sqlalchemy import create_engine, text
from config import *
from dmutils import save_result

pd.set_option("display.width", 170, "display.max_columns", 40)
ENGINE = create_engine(DB_URL)
FIG_DIR = OUT_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

def q(sql):
    """Chạy một câu SQL, trả DataFrame."""
    return pd.read_sql(text(sql), ENGINE)

def rec(df):
    """DataFrame -> list dict an toàn cho JSON (NaN -> None, categorical -> str)."""
    d = df.copy()
    for c in d.columns:
        if str(d[c].dtype) == "category": d[c] = d[c].astype(str)
    d = d.astype(object).where(d.notna(), None)
    return d.to_dict("records")

def wilson(k, n, alpha=0.05):
    """Khoảng Wilson cho tỉ lệ k/n -> (rate, lo, hi). Cùng công thức với mart.wilson_lo/hi."""
    if n == 0: return (np.nan, np.nan, np.nan)
    z = norm.ppf(1 - alpha / 2); p = k / n; den = 1 + z**2 / n
    ctr = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return (p, max(0.0, ctr - half), min(1.0, ctr + half))

def low_table(df, by, order=None):
    """Bảng tỉ lệ app điểm thấp theo nhóm: n, k, rate, Wilson lo/hi, enough_n (n >= MIN_CELL_N)."""
    by = [by] if isinstance(by, str) else list(by)
    keys = by + ([order] if order else [])
    g = df.groupby(keys, observed=True)["is_low"].agg(n="size", k="sum").reset_index()
    w = pd.DataFrame([wilson(int(k), int(n)) for k, n in zip(g["k"], g["n"])], columns=["rate", "lo", "hi"])
    g = pd.concat([g, w], axis=1)
    g["enough_n"] = g["n"] >= MIN_CELL_N
    return g.sort_values(order if order else "rate", ascending=bool(order)).reset_index(drop=True)

## 1. Tham số khớp `mart.params` (nguồn duy nhất do `run_all.py` đồng bộ)

In [ ]:
prm = q("SELECT low_rating_threshold, min_store_reviews, min_cell_n FROM mart.params").iloc[0]
assert float(prm.low_rating_threshold) == LOW_RATING_THRESHOLD
assert int(prm.min_store_reviews) == MIN_STORE_REVIEWS and int(prm.min_cell_n) == MIN_CELL_N
print("LOW_RATING_THRESHOLD =", LOW_RATING_THRESHOLD, "| MIN_STORE_REVIEWS =", MIN_STORE_REVIEWS, "| MIN_CELL_N =", MIN_CELL_N)

## 2. Nạp `dw.fact_app` kèm các dimension và đếm "phễu" (ba con số *n* đi vào báo cáo)

In [ ]:
apps = q("""
SELECT f.app_key, c.category, f.rating, f.store_reviews, f.size_mb, f.price,
       pr.price_type, pr.price_band, pr.band_order AS price_order,
       sz.size_band,  sz.band_order  AS size_order,
       ins.install_band, ins.band_order AS install_order,
       d.age_band, d.age_order, d.age_days
FROM dw.fact_app f
JOIN dw.dim_category c   USING (category_key)
JOIN dw.dim_price    pr  USING (price_key)
JOIN dw.dim_size     sz  USING (size_key)
JOIN dw.dim_install  ins USING (install_key)
JOIN dw.dim_date     d   USING (update_key)
""")
rated = apps[apps["rating"].notna()]
base = rated[rated["store_reviews"] >= MIN_STORE_REVIEWS].copy()
base["is_low"] = base["rating"] <= LOW_RATING_THRESHOLD
funnel = {"n_app_dedup": len(apps), "n_missing_rating": int(apps["rating"].isna().sum()), "n_rated": len(rated),
          "n_below_min_reviews": len(rated) - len(base), "n_analysis": len(base),
          "n_low": int(base["is_low"].sum()), "low_rate_global": float(base["is_low"].mean())}
print(funnel)
n_unknown = {c: int((base[c] == "Unknown").sum()) for c in ["category", "price_band", "size_band", "install_band", "age_band"]}
print("số dòng rơi vào band 'Unknown':", n_unknown)
save_result("nb03_funnel", {**funnel, "unknown_band_counts": n_unknown})

## 3. Nhóm thiếu Rating có giống nhóm còn lại không?
Quyết định #4: không điền Rating thiếu. Cần nêu khả năng thiếu **không ngẫu nhiên** (MNAR): so phân bố band lượt cài và trung vị `store_reviews` của hai nhóm.

In [ ]:
apps["has_rating"] = apps["rating"].notna()
cmp_install = (apps.groupby(["install_order", "install_band", "has_rating"]).size().unstack("has_rating", fill_value=0))
cmp_install = (cmp_install / cmp_install.sum()).round(3).rename(columns={False: "share_missing", True: "share_rated"})
cmp_reviews = apps.groupby("has_rating")["store_reviews"].median().rename({False: "median_missing", True: "median_rated"})
print(cmp_install, "\n", cmp_reviews)
save_result("nb03_missing_rating_vs_rated", {"install_share": rec(cmp_install.reset_index()), "median_store_reviews": cmp_reviews.to_dict(),
                                              "n_missing": funnel["n_missing_rating"], "n_rated": funnel["n_rated"]})

## 4. Năm bảng chính (Wilson 95%; ô `n < MIN_CELL_N` hiển thị nhưng **không diễn giải**)

In [ ]:
tabs = {
  "category":     low_table(base, "category"),
  "price_band":   low_table(base, "price_band", "price_order"),
  "size_band":    low_table(base, "size_band", "size_order"),
  "install_band": low_table(base, "install_band", "install_order"),
  "age_band":     low_table(base, "age_band", "age_order"),
}
for k, t in tabs.items():
    print(f"\n== {k}  (số ô: {len(t)}, ô đủ n: {int(t['enough_n'].sum())}) =="); print(t.round(3).to_string(index=False))
    save_result(f"nb03_low_rate_by_{k}", rec(t))

## 5. Đối chiếu với view của B (`mart.v_lowrated_by_category`): n và số app điểm thấp phải khớp
Lệch nghĩa là một trong hai bên sai (cấu hình ngưỡng, join, hay lọc). Tìm ra trước khi viết kết luận.

In [ ]:
v = q("SELECT category, n_apps, n_low FROM mart.v_lowrated_by_category")
chk = tabs["category"].merge(v, on="category", how="outer", indicator=True)
assert (chk["_merge"] == "both").all(), chk[chk["_merge"] != "both"]
assert (chk["n"] == chk["n_apps"]).all() and (chk["k"] == chk["n_low"]).all()
print("Khớp với mart.v_lowrated_by_category:", len(chk), "danh mục")

## 6. Trung bình và trung vị điểm theo danh mục (Rating dồn về đầu cao nên trung bình bị kéo)

In [ ]:
mm = base.groupby("category")["rating"].agg(n="size", mean="mean", median="median").reset_index()
mm["gap"] = (mm["mean"] - mm["median"]).abs(); mm["enough_n"] = mm["n"] >= MIN_CELL_N
print(mm.sort_values("median").round(3).head(10).to_string(index=False))
n5 = int((base["rating"] == 5.0).sum())
print("app đúng 5.0 (sau ngưỡng):", n5, "/", len(base))
save_result("nb03_mean_median_by_category", rec(mm)); save_result("nb03_exact_5", {"n_exact_5": n5, "n_analysis": len(base)})

## 7. Hai phép cắt chéo (có *n* từng ô; ô nhỏ không diễn giải)

In [ ]:
cross_price = low_table(base, ["category", "price_type"])
cross_age_inst = low_table(base, ["install_band", "age_band"])
print("category x Free/Paid:", len(cross_price), "ô,", int(cross_price["enough_n"].sum()), "đủ n")
print("install x age:", len(cross_age_inst), "ô,", int(cross_age_inst["enough_n"].sum()), "đủ n")
print(cross_price[cross_price["price_type"] == "Paid"].sort_values("n", ascending=False).head(8).round(3).to_string(index=False))
save_result("nb03_cross_category_pricetype", rec(cross_price)); save_result("nb03_cross_install_age", rec(cross_age_inst))

## 8. Biểu đồ (tiêu đề là kết luận, có *n*; chỉ vẽ ô đủ n)

In [ ]:
t = tabs["category"]; t = t[t["enough_n"]].sort_values("rate")
fig, ax = plt.subplots(figsize=(7, max(4, 0.28 * len(t))))
ax.barh(t["category"], t["rate"] * 100, xerr=[(t["rate"] - t["lo"]) * 100, (t["hi"] - t["rate"]) * 100], color="#4C78A8", ecolor="#555")
ax.axvline(funnel["low_rate_global"] * 100, color="k", ls="--", lw=1, label=f"toàn cục {funnel['low_rate_global']*100:.1f}%")
top = t.iloc[-1]
ax.set_title(f"{top['category']} có tỉ lệ app điểm thấp cao nhất ({top['rate']*100:.1f}%, n={int(top['n'])}) [sơ bộ, xem nb05]", fontsize=9)
ax.set_xlabel(f"% app có Rating ≤ {LOW_RATING_THRESHOLD} (Wilson 95%), chỉ danh mục n ≥ {MIN_CELL_N}"); ax.legend(); fig.tight_layout()
fig.savefig(FIG_DIR / "nb03_lowrate_by_category.png", dpi=130); plt.show()

## 9. Kết luận của nb03 (C điền SAU khi nb05 chạy xong)
Mỗi dòng: một kết luận **kèm n và khoảng Wilson**, dùng "đi cùng". Kết luận nào nb05 báo "Đổi" thì ghi "sơ bộ" hoặc bỏ. Ví dụ khung:
- Danh mục … có tỉ lệ app điểm thấp cao nhất (…%, Wilson …–…, n = …); nb05: giữ/đổi vì …
- App trả phí … so với miễn phí …
- App cập nhật cách đây > 2 năm … so với <= 3 tháng …